In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### Helper function to add functions to a class 

In [2]:
def add_to_class(Class):  #@save
    """Register functions as methods in created class."""
    def wrapper(obj):
        setattr(Class, obj.__name__, obj)
    return wrapper

In [3]:
class ActivationFunctions:
    def act_sigmoid(self, z):      
        output = np.empty_like(z, dtype=float) 
        pos_mask = z >= 0
        neg_mask = ~pos_mask
        output[pos_mask] = 1 / (1 + np.exp(-z[pos_mask]))
        output[neg_mask] = np.exp(z[neg_mask]) / (1 + np.exp(z[ng_mask]))
        return output
        
    def act_tanh(self,z):
        output = 2 * self.act_sigmoid(2*z) -1 
        return output
        
    def act_relu(self, z):
        output = np.where(z>0, z, 0)
        return output

    def ddz_sigmoid(self, z):
        """ derivative of the sigmoid function with respect to z """
        ddz = self.act_sigmoid(z) * (1 - self.act_sigmoid(z))
        return ddz

    def ddz_tanh(self,z):
        """ derivative of the hyperbolic tangent function with respect to z """
        ddz = 1 - self.act_tanh(z)
        return ddz

    def ddz_relu(self,z):
        """ derivative of the ReLU function with respect to z """
        ddz = np.where(z > 0, 1, 0)
        return ddz

AF = ActivationFunctions()

# Single lookup of (activation, derivative) pairs, shared by Layer.__init__ and Layer.layer_activation
ACTIVATIONS = {
    'relu': (AF.act_relu, AF.ddz_relu),
    'tanh': (AF.act_tanh, AF.ddz_tanh),
    'sigmoid': (AF.act_sigmoid, AF.ddz_sigmoid)
}


In this exercise we will create a neural network with the following features (in order of implementation):</p>


* Choice of $\tanh(\cdot)$, $\sigma(\cdot)$ and $\text{ReLU}(\cdot)$ (default) activation. 
* Arbitrary number of nodes in a layer.
* Layer weights initialized with *Xavier Initialization* (a technique to keep the weights from becoming too big or too small).
* Arbitrary number of layers.
* Binary classification only, MSE loss.
* Backpropagation, SGD optimization.
* Training update with arbitrary number of training epochs.

In [4]:
class NeuralNetwork():

    def __init__(self,data_features=None,data_labels=None):
        if data_features is not None and data_labels is not None:
            self.load_data(data_features,data_labels)
        self.prediction = None

    def load_data(self,data_features,data_labels):
        self.features = data_features
        self.input_size = data_features[0].size #assumes np array where rows are data instances, columns are features
        self.labels = data_labels
        self.output_size = data_labels[0].size #same assumption, for labels.


Next, create a $\texttt{Layer}$ subclass to handle the creation of arbitrarily sized layers. We will need to include the function to initialize the weights with a zero mean gaussian and a variance to be determined by xavier initialization later (with a possible seed).

In [5]:
@add_to_class(NeuralNetwork)
class Layer():
    def __init__(self, n_nodes, n_input, weight_var, init_seed=0, layer_activation='ReLU'):
        # Can expect standard labels for activation functions: 'ReLU', 'tanh', or 'sigmoid'
        self.weights = self.weights_initialization(n_nodes, n_input, weight_var, init_seed)
        self.activation = None
        self.output = None

    def weights_initialization(self, n_nodes, n_input, weight_var, init_seed):
        np.random.seed(init_seed)
        #initialize weight matrix with 0 mean, np.sqrt(weight_var) as standard deviation, and 
        #the output array of size (n_input, n_nodes)
        init_weights = np.random.normal(0, np.sqrt(weight_var), (n_input, n_nodes))
        return init_weights

Add to the $\texttt{Layer}$ subclass a way to handle the selection of an activation function for each layer, and have it default to $\text{ReLU}(\cdot)$

In [6]:
@add_to_class(NeuralNetwork.Layer)
def layer_activation(self, activation_type = 'ReLU'):
    a = ActivationFunctions()
    self.activation_type = activation_type
    if activation_type == 'ReLU':
        self.activation = a.act_ReLU
        self.der_act = a.ddx_ReLU
    elif activation_type == 'tanh':
        self.activation = a.act_tanh
        self.der_act = a.ddx_tanh
    else:
        self.activation = a.act_sigmoid
        self.der_act = a.ddx_sigmoid

With the $\texttt{Layer}$ subclass, we can write a function to initialize the network's layers. Since we know the number of nodes for each layer, we can determine the initial weight variance for numerical stability.

In [7]:
@add_to_class(NeuralNetwork)
def init_network(self, layer_nodes_tuple, activation_tuple):
    #output size is 1 since we only care about binary classification for now
    #layer_nodes_tuple contains the dimensions of the hidden layers
    #activation_tuple contains the activation function for each of the layers
    #For example, for NN with 3 hidden layers containing 8, 6, & 4 nodes respectively,
    #all using ReLU, and one outer layer using Sigmoid
    #the layer_nodes_tuple will be (8,6,5) and activation_tuple will be ('ReLU', 'ReLU', 'ReLU', 'sigmoid')
    
    self.layers = []
    #get all layer sizes - input, hidden, and output - in one array
    all_sizes = [self.input_size] + list(layer_nodes_tuple) + [self.output_size]

    #calculate variances for Xavier initialization for each layer
    layer_weight_variances = [2.0 / (all_sizes[i] + all_sizes[i+1]) for i in range(len(all_sizes) - 1)]
    
    #create each layer's nodes and activations and then add to list of layers
    for i, (n_node, activation_str) in enumerate(zip(list(layer_nodes_tuple) + [self.output_size], activation_tuple)):
        layer = self.Layer(n_node, all_sizes[i], layer_weight_variances[i])
        layer.layer_activation(activation_str)
        self.layers.append(layer)

### Perform feedforward and backpropagation computation. 
With the network initialized, we can perform computations, both feedforwad and backpropagation. We will start with the feedforward function and cost function. With a small amount of foresight, we can also write a function for the derivative of the cost w.r.t. the output layer output.
¶

In [8]:
@add_to_class(NeuralNetwork)
def feedforward(self, features, label, verbose=False):
    ## layer weights should hold respective activation and output
    current_input = features
    for layer in self.layers:
        layer.output = np.einsum('f,fl->l',current_input,layer.weights)        # np.einsum is equivalent to dot product. a = input · W
        current_input = layer.activation(layer.output)       # o = activation(a)
        layer.output = current_input                         # store activated output
    self.prediction = current_input


@add_to_class(NeuralNetwork)
def compute_cost(self, label, verbose=False):
    assert self.prediction is not None, "Prediction unavailable. Perform feedforward first."
    cost = np.mean((self.prediction - label) ** 2)
    return cost

@add_to_class(NeuralNetwork)
def compute_dcost_dout(self, label):
    assert self.prediction is not None, "Prediction unavailable. Perform feedforward first."
    dcost_dout = 2 * (self.prediction - label)
    return dcost_dout


Now we will write a function to perform the backpropagation. Ensure the weights of each layer are updated, but the error vector is kept unperturbed, i.e. not recomputed with new weights.

In [9]:
@add_to_class(NeuralNetwork)
def backpropagation(self,features,label,verbose=False):
    self.feedforward(features, label)
    error = self.compute_dcost_dout(label)
    for i in reversed(range(len(self.layers))):
        layer = self.layers[i]
        if i == 0:
            layer_input = features
        else:
            layer_input = self.layers[i - 1].output
        pre_activation = layer_input@layer.weights
        delta = error * layer.der_act(pre_activation)
        dW = np.outer(layer_input, delta)
        layer.weights -= self.lr*dW
        error = delta@layer.weights.T ### different matrix calculus conventions


Finally, to train the network we will update over N epochs. One epoch samples the entire dataset for update. To avoid a bias in updates, you should sample the dataset randomly without replacement.

In [10]:
@add_to_class(NeuralNetwork)
def train_network(self, N_epochs):
    for ix in range(N_epochs):
        ### one epoch ###
        indices = np.random.permutation(len(self.features)) ### this is stochastic processs <- approximates the gradient
        for i in indices:
            self.backpropagation(self.features[i], self.labels[i])
            ### stochastic gradient descent ##